# 🐉 Imagen → 3D con Hunyuan3D-2 (gratis, calidad alta)Convierte **una imagen de personaje** en un **modelo 3D** (`.glb`) en la GPU **gratis** de Colab.Ya trae **todas las correcciones** que fuimos encontrando:- Usa el modelo **completo** `tencent/Hunyuan3D-2` (el `mini` daba *"Model path not found"*).- Arregla el `numpy` para que `scipy` no rompa (`_blas_supports_fpe`).- **Quita el fondo solo** (funciona con cualquier imagen, no hace falta PNG transparente).- Genera **solo la forma** (malla gris), que es lo que entra en la T4 (~6 GB). El color es un paso aparte más pesado.## Cómo usarlo (celda por celda)1. **GPU**: menú `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.2. **Celda 1**: instalar (tarda ~5–8 min la primera vez). **Cuando termine, reiniciá la sesión** (te lo recuerda al final: `Entorno de ejecución` → `Reiniciar sesión`).3. **Celda 2**: subir tu imagen.4. **Celda 3**: generar el modelo 3D (la 1ª vez baja ~10 GB de pesos, esperá).5. **Celda 4**: descargar el `.glb`.> Después animás el `.glb` (caminar/correr) en https://imagen-a-3d-triposr.vercel.app → botón **“Animar un modelo 3D”**, o me lo pasás y le pongo las animaciones mocap.

### Celda 1 — Instalar Hunyuan3D-2  ·  al terminar: **Reiniciar sesión**

In [ ]:
!nvidia-smi -Limport osos.chdir('/content')if not os.path.isdir('/content/Hunyuan3D-2'):    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.gitos.chdir('/content/Hunyuan3D-2')# Dependencias para la generacion de FORMA (no hace falta compilar las de textura)!pip install -q ninja!pip install -q diffusers transformers accelerate trimesh omegaconf einops opencv-python-headless huggingface_hub!pip install -q rembg onnxruntime            # para quitar el fondo automaticamente!pip install -q -e . 2>&1 | tail -3          # instala el paquete hy3dgen (ignora warnings de version)# Alinea numpy para que scipy no rompa con "_blas_supports_fpe"!pip install -q -U "numpy>=2.1"import torchprint('\ntorch:', torch.__version__, '| GPU:', torch.cuda.is_available())print('\n✅ Instalado. AHORA hacé:  Entorno de ejecución → Reiniciar sesión')print('   (es por el cambio de numpy). Después seguí con la Celda 2.')

### Celda 2 — Subir tu imagenCualquier imagen sirve (el fondo se quita solo). Mejor si el personaje está **de frente** y entero.

In [ ]:
from google.colab import filesfrom PIL import Imageup = files.upload()IMG = list(up.keys())[0]im = Image.open(IMG)print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)print('✅ Lista. Seguí con la Celda 3.')

### Celda 3 — Generar el modelo 3D (forma)La **primera vez** descarga los pesos del modelo (~10 GB), puede tardar varios minutos. `octree_resolution` más alto = más detalle (y más VRAM/tiempo). Si da *CUDA out of memory*, bajalo a `192`.

In [ ]:
import os, torchos.chdir('/content/Hunyuan3D-2')from PIL import Imagefrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline# Modelo COMPLETO (mejor calidad). El 'mini' NO se usa: su subfolder por defecto no coincide.print('Cargando el modelo (la 1a vez baja ~10 GB)...')pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')print('✅ Modelo cargado.')img = Image.open(IMG)# Quitar el fondo si no es transparente -> mejora mucho la reconstruccionif img.mode != 'RGBA':    try:        from hy3dgen.rembg import BackgroundRemover        img = BackgroundRemover()(img.convert('RGB'))        print('✅ Fondo quitado automaticamente.')    except Exception as e:        print('⚠️ No se pudo quitar el fondo (', e, '). Uso la imagen tal cual; para mejor calidad subí un PNG transparente.')        img = img.convert('RGBA')mesh = pipe(image=img, num_inference_steps=30, octree_resolution=256,            generator=torch.manual_seed(0))[0]OUT = '/content/hunyuan_mesh.glb'mesh.export(OUT)print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024, 1)) + ' KB')      if os.path.exists(OUT) else '❌ no se generó, copiame el error de arriba')

### Celda 4 — Descargar el `.glb`

In [ ]:
from google.colab import filesfiles.download('/content/hunyuan_mesh.glb')

---### Si algo falla- **`CUDA out of memory`** (Celda 3): bajá `octree_resolution` a `192`, o `Entorno de ejecución → Reiniciar sesión` y probá de nuevo.- **Se ve deforme / con el fondo pegado**: la quita-fondo no acertó; subí un **PNG con fondo transparente**, de frente.- **Error de `numpy`/`scipy`** al importar: te faltó **Reiniciar sesión** después de la Celda 1. Reiniciá y corré Celda 2 y 3.- Este modo genera **solo la forma** (malla gris, sin color). La geometría ya es muy buena. Si querés **color/textura**, avisame y armamos el paso extra (más pesado).- Cualquier error rojo, copiámelo y lo ajustamos.